# Modul 10: Deep Learning Sekuensial — LSTM & Multi-Step Seq2Seq (PyTorch)
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 10 & 11 (Minggu 11–12):** Menyiapkan tensor sekuensial 3D $(B, T, d_x)$ bebas *data leakage*, melatih arsitektur LSTM multi-output (24 jam simultan) pada data PLTS, serta memantau kurva konvergensi *loss*.

In [ ]:
# !pip install -q torch scikit-learn numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Perangkat komputasi aktif:", device)

## 1. Simulasi Data Daya PLTS Multivariat & Pembentukan Tensor 3D (Scaling Hanya di Train!)

In [ ]:
n_hours = 24 * 45  # 45 hari
t = np.arange(n_hours)
hour = t % 24

irradiance = np.where((hour >= 6) & (hour <= 18), 900 * np.sin(np.pi * (hour - 6) / 12), 0.0)
cloud = np.clip(1.0 - 0.35 * (np.random.rand(n_hours) < 0.15), 0.3, 1.0)
irradiance = irradiance * cloud
temp_panel = 25 + 0.03 * irradiance + np.random.normal(0, 1.0, n_hours)
pv_power = np.clip(0.52 * irradiance * (1 - 0.004 * (temp_panel - 25)) + np.random.normal(0, 8, n_hours), 0, None)

data = np.column_stack([pv_power, irradiance, temp_panel])
split = int(len(data) * 0.8)

# ANTI-DATA LEAKAGE: fit scaler HANYA pada data train
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(data[:split])
test_scaled = scaler.transform(data[split:])

def make_windows(arr, lookback=24, horizon=24):
    X_list, y_list = [], []
    for i in range(len(arr) - lookback - horizon + 1):
        X_list.append(arr[i : i + lookback, :])
        y_list.append(arr[i + lookback : i + lookback + horizon, 0])  # target: pv_power
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

X_tr, y_tr = make_windows(train_scaled, lookback=24, horizon=24)
X_te, y_te = make_windows(test_scaled, lookback=24, horizon=24)
print(f"Tensor Train: X={X_tr.shape}, y={y_tr.shape} | Test: X={X_te.shape}, y={y_te.shape}")

## 2. Arsitektur LSTM Multi-Step Forecaster (24 Jam Simultan) & Pelatihan

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=48, num_layers=2, horizon=24):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.15)
        self.fc = nn.Linear(hidden_dim, horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = LSTMForecaster(input_dim=3, hidden_dim=48, num_layers=2, horizon=24).to(device)
print("Total parameter latih:", sum(p.numel() for p in model.parameters() if p.requires_grad))

loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=32, shuffle=False)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
criterion = nn.MSELoss()

losses = []
for epoch in range(1, 26):
    model.train()
    ep_loss = 0.0
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item() * len(bx)
    losses.append(ep_loss / len(X_tr))
    if epoch % 5 == 0:
        print(f"Epoch {epoch:02d} | Train MSE Loss: {losses[-1]:.5f}")

## 3. Evaluasi Multi-Horizon 24 Jam & Visualisasi

In [ ]:
model.eval()
with torch.no_grad():
    pred_scaled = model(torch.from_numpy(X_te).to(device)).cpu().numpy()

# Inverse scale khusus kolom 0 (pv_power)
pv_min, pv_max = scaler.data_min_[0], scaler.data_max_[0]
pred_kw = pred_scaled * (pv_max - pv_min) + pv_min
true_kw = y_te * (pv_max - pv_min) + pv_min

rmse = np.sqrt(np.mean((true_kw - pred_kw) ** 2))
print(f"Test Multi-Step RMSE (24 jam): {rmse:.2f} kW")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(losses, lw=2, color='navy')
axes[0].set_title('Kurva Konvergensi Training Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].grid(True)

axes[1].plot(true_kw[0], 'k-o', ms=4, label='Aktual PV (24 Jam)')
axes[1].plot(pred_kw[0], 'r--s', ms=4, label='Prediksi LSTM 24-Step')
axes[1].set_title('Contoh Peramalan Daya PLTS 24 Jam Simultan')
axes[1].set_ylabel('Daya (kW)')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()

## 4. Tugas & Pertanyaan Analisis (C4–C6)
1. **[C4]** Ganti `nn.LSTM` menjadi `nn.GRU` dengan ukuran `hidden_dim` sama. Bandingkan jumlah parameter, kecepatan pelatihan, dan RMSE-nya!
2. **[C6]** Tambahkan lapisan `nn.Conv1d` sebelum LSTM (arsitektur hibrida **1D-CNN + LSTM**) untuk mengekstrak pola awan lokal sebelum pemodelan sekuensial.